# Actividad 2. Modelo de probabilidad de pago

Se estima la probabilidad de que cada cuenta por cobrar alcance el pago total dentro de 120 días. La probabilidad se utiliza para priorizar la gestión y calcular la exposición esperada.

In [32]:
from pathlib import Path

import numpy as np
import pandas as pd

from src.analytical_table_builder import AnalyticalTableBuilder
from src.database import SQLiteDatabase


project_path = Path.cwd()

if project_path.name == "notebooks":
    project_path = project_path.parent

database = SQLiteDatabase(
    project_path / "data" / "base_datos_historica.db"
)

table_builder = AnalyticalTableBuilder(
    database=database,
    sql_path=project_path / "src" / "sql" / "sabana_analitica.sql",
)

modeling_data = table_builder.build()

date_columns = [
    "creation_date",
    "last_payment_date",
    "reference_date",
]

for column_name in date_columns:
    modeling_data[column_name] = pd.to_datetime(
        modeling_data[column_name],
        errors="coerce",
    )

modeling_data.shape

(21739, 25)

In [33]:
horizon_results = []

for horizon_days in [90, 120, 180, 270, 360]:
    paid_within_horizon = (
        modeling_data["is_fully_paid"].eq(1)
        & modeling_data["days_creation_to_last_payment"].between(
            0,
            horizon_days,
            inclusive="both",
        )
    )

    has_complete_observation_window = (
        modeling_data["age_days"] >= horizon_days
    )

    eligible_records = (
        paid_within_horizon
        | has_complete_observation_window
    )

    target_values = paid_within_horizon.loc[
        eligible_records
    ].astype(int)

    horizon_results.append(
        {
            "horizon_days": horizon_days,
            "eligible_records": eligible_records.sum(),
            "excluded_records": (~eligible_records).sum(),
            "paid_records": target_values.sum(),
            "not_paid_within_horizon": (
                target_values.eq(0).sum()
            ),
            "payment_rate": round(
                target_values.mean() * 100,
                2,
            ),
        }
    )

horizon_summary = pd.DataFrame(horizon_results)
horizon_summary

,horizon_days,eligible_records,excluded_records,paid_records,not_paid_within_horizon,payment_rate
0,90,21739,0,7942,13797,36.53
1,120,21276,463,10325,10951,48.53
2,180,20277,1462,13733,6544,67.73
3,270,18813,2926,16485,2328,87.63
4,360,17423,4316,17321,102,99.41


## 1. Variable objetivo

El objetivo toma el valor `1` cuando la obligación se paga totalmente dentro de 120 días y `0` cuando no alcanza el pago total en ese periodo.

Se excluyen las obligaciones no pagadas con menos de 120 días de observación. La fecha del último pago se usa como aproximación del momento de pago total.

In [34]:
horizon_days = 120

paid_within_horizon = (
    modeling_data["is_fully_paid"].eq(1)
    & modeling_data["days_creation_to_last_payment"].between(
        0,
        horizon_days,
        inclusive="both",
    )
)

eligible_mask = (
    paid_within_horizon
    | modeling_data["age_days"].ge(horizon_days)
)

model_data = modeling_data.loc[eligible_mask].copy()

model_data["target_paid_120d"] = (
    paid_within_horizon.loc[eligible_mask]
    .astype(int)
)

target_validation = (
    model_data["target_paid_120d"]
    .value_counts()
    .rename_axis("target_paid_120d")
    .reset_index(name="records")
)

target_validation["percentage"] = (
    target_validation["records"]
    / target_validation["records"].sum()
    * 100
).round(2)

print(f"Registros elegibles: {len(model_data):,}")
print(f"Registros excluidos: {(~eligible_mask).sum():,}")

target_validation

Registros elegibles: 21,276
Registros excluidos: 463


,target_paid_120d,records,percentage
0,0,10951,51.47
1,1,10325,48.53


## 2. Variables y control de fuga

Se utilizan producto, código de transacción, valor original y fecha de creación. Se excluyen saldo, valor pagado, fecha de pago y estado final porque contienen información posterior a la creación.

`num_cta` se usa solamente para separar las muestras y evitar que una misma cuenta aparezca en entrenamiento y validación.

In [35]:
model_data["log_original_amount"] = np.log1p(
    model_data["vlr_original"]
)

model_data["creation_year"] = (
    model_data["creation_date"].dt.year
)

model_data["creation_month"] = (
    model_data["creation_date"].dt.month
)

model_data["creation_weekday"] = (
    model_data["creation_date"].dt.dayofweek
)

model_data["cod_apli_prod"] = (
    model_data["cod_apli_prod"].astype(str)
)

model_data["cod_trn"] = (
    model_data["cod_trn"].astype(str)
)

feature_columns = [
    "cod_apli_prod",
    "cod_trn",
    "log_original_amount",
    "creation_year",
    "creation_month",
    "creation_weekday",
]

X = model_data[feature_columns].copy()
y = model_data["target_paid_120d"].copy()
groups = model_data["num_cta"].copy()

X.shape, y.shape

((21276, 6), (21276,))

In [36]:
from sklearn.model_selection import GroupShuffleSplit


group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_index, test_index = next(
    group_split.split(
        X,
        y,
        groups=groups,
    )
)

X_train = X.iloc[train_index].copy()
X_test = X.iloc[test_index].copy()

y_train = y.iloc[train_index].copy()
y_test = y.iloc[test_index].copy()

groups_train = groups.iloc[train_index]
groups_test = groups.iloc[test_index]

In [37]:
account_overlap = set(groups_train).intersection(
    set(groups_test)
)

split_summary = pd.DataFrame(
    {
        "dataset": ["train", "test"],
        "records": [
            len(X_train),
            len(X_test),
        ],
        "accounts": [
            groups_train.nunique(),
            groups_test.nunique(),
        ],
        "payment_rate": [
            round(y_train.mean() * 100, 2),
            round(y_test.mean() * 100, 2),
        ],
    }
)

print(f"Cuentas presentes en ambos conjuntos: {len(account_overlap)}")
split_summary

Cuentas presentes en ambos conjuntos: 0


,dataset,records,accounts,payment_rate
0,train,17026,640,48.64
1,test,4250,160,48.07


## 3. Comparación de modelos

La regresión logística se compara primero con una probabilidad general de referencia. Después se incorpora un bosque aleatorio para verificar si una alternativa más compleja mejora el resultado.

In [38]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


categorical_features = [
    "cod_apli_prod",
    "cod_trn",
]

numeric_features = [
    "log_original_amount",
    "creation_year",
    "creation_month",
    "creation_weekday",
]

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features,
        ),
        (
            "numeric",
            numeric_transformer,
            numeric_features,
        ),
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

dummy_model = DummyClassifier(
    strategy="prior",
    random_state=42,
)

logistic_model.fit(X_train, y_train)
dummy_model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'prior'
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](2,)","[0.51,0.49]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](6,)","['cod_apli_prod','cod_trn','log_original_amount','creation_year', 'creation_month','creation_weekday']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,6
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [39]:
def evaluate_model(
    model_name,
    model,
    features,
    target,
    threshold=0.50,
):
    """Calcula métricas de clasificación y probabilidad."""
    predicted_probability = model.predict_proba(features)[:, 1]

    predicted_class = (
        predicted_probability >= threshold
    ).astype(int)

    return {
        "model": model_name,
        "roc_auc": roc_auc_score(
            target,
            predicted_probability,
        ),
        "average_precision": average_precision_score(
            target,
            predicted_probability,
        ),
        "brier_score": brier_score_loss(
            target,
            predicted_probability,
        ),
        "accuracy": accuracy_score(
            target,
            predicted_class,
        ),
        "precision": precision_score(
            target,
            predicted_class,
            zero_division=0,
        ),
        "recall": recall_score(
            target,
            predicted_class,
            zero_division=0,
        ),
        "f1_score": f1_score(
            target,
            predicted_class,
            zero_division=0,
        ),
    }

In [40]:
model_comparison = pd.DataFrame(
    [
        evaluate_model(
            model_name="dummy_prior",
            model=dummy_model,
            features=X_test,
            target=y_test,
        ),
        evaluate_model(
            model_name="logistic_regression",
            model=logistic_model,
            features=X_test,
            target=y_test,
        ),
    ]
).round(4)

model_comparison

,model,roc_auc,average_precision,brier_score,accuracy,precision,recall,f1_score
0,dummy_prior,0.5000,0.4807,0.2497,0.5193,0.0000,0.0000,0.0000
1,logistic_regression,0.7139,0.7314,0.2128,0.6659,0.6728,0.5937,0.6308


La regresión logística supera la referencia inicial. Las métricas muestran capacidad moderada para ordenar las obligaciones según su probabilidad de pago.

In [41]:
from sklearn.ensemble import RandomForestClassifier


random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=20,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

random_forest_model.fit(
    X_train,
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['cod_apli_prod','cod_trn','log_original_amount','creation_year', 'creation_month','creation_weekday']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).B

In [42]:
model_comparison = pd.DataFrame(
    [
        evaluate_model(
            model_name="dummy_prior",
            model=dummy_model,
            features=X_test,
            target=y_test,
        ),
        evaluate_model(
            model_name="logistic_regression",
            model=logistic_model,
            features=X_test,
            target=y_test,
        ),
        evaluate_model(
            model_name="random_forest",
            model=random_forest_model,
            features=X_test,
            target=y_test,
        ),
    ]
).round(4)

model_comparison

,model,roc_auc,average_precision,brier_score,accuracy,precision,recall,f1_score
0,dummy_prior,0.5000,0.4807,0.2497,0.5193,0.0000,0.0000,0.0000
1,logistic_regression,0.7139,0.7314,0.2128,0.6659,0.6728,0.5937,0.6308
2,random_forest,0.7194,0.7387,0.2158,0.6784,0.7172,0.5463,0.6202


El bosque aleatorio mejora ligeramente la discriminación, pero no mejora la calidad de las probabilidades de forma relevante. La decisión final se confirma con validación cruzada agrupada por cuenta.

In [43]:
from sklearn.model_selection import GroupKFold, cross_validate


group_cross_validation = GroupKFold(n_splits=5)

candidate_models = {
    "logistic_regression": logistic_model,
    "random_forest": random_forest_model,
}

cross_validation_results = []

for model_name, model in candidate_models.items():
    scores = cross_validate(
        estimator=model,
        X=X,
        y=y,
        groups=groups,
        cv=group_cross_validation,
        scoring={
            "roc_auc": "roc_auc",
            "average_precision": "average_precision",
            "brier_score": "neg_brier_score",
        },
        n_jobs=-1,
    )

    cross_validation_results.append(
        {
            "model": model_name,
            "roc_auc_mean": scores["test_roc_auc"].mean(),
            "roc_auc_std": scores["test_roc_auc"].std(),
            "average_precision_mean": (
                scores["test_average_precision"].mean()
            ),
            "average_precision_std": (
                scores["test_average_precision"].std()
            ),
            "brier_score_mean": (
                -scores["test_brier_score"].mean()
            ),
            "brier_score_std": (
                scores["test_brier_score"].std()
            ),
        }
    )

cross_validation_summary = pd.DataFrame(
    cross_validation_results
).round(4)

cross_validation_summary

,model,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,brier_score_mean,brier_score_std
0,logistic_regression,0.7028,0.0086,0.7186,0.0179,0.2180,0.0033
1,random_forest,0.7078,0.0048,0.7256,0.0115,0.2185,0.0021


## 4. Selección del modelo

Los dos modelos presentan resultados similares. Se selecciona la regresión logística porque mantiene un Brier Score ligeramente mejor y es más sencilla de interpretar y explicar.

In [44]:
selected_model = logistic_model

feature_names = (
    selected_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    selected_model
    .named_steps["classifier"]
    .coef_[0]
)

coefficient_summary = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coefficients,
    }
)

coefficient_summary["absolute_coefficient"] = (
    coefficient_summary["coefficient"].abs()
)

coefficient_summary["effect"] = np.where(
    coefficient_summary["coefficient"] > 0,
    "aumenta_probabilidad",
    "disminuye_probabilidad",
)

coefficient_summary = coefficient_summary.sort_values(
    "absolute_coefficient",
    ascending=False,
)

coefficient_summary.head(15)

,feature,coefficient,absolute_coefficient,effect
61,categorical__cod_trn_75,-2.313213,2.313213,disminuye_probabilidad
51,categorical__cod_trn_416,-2.078583,2.078583,disminuye_probabilidad
69,numeric__creation_year,1.803127,1.803127,aumenta_probabilidad
70,numeric__creation_month,1.491375,1.491375,aumenta_probabilidad
62,categorical__cod_trn_788,-1.249690,1.249690,disminuye_probabilidad
42,categorical__cod_trn_301,-1.138355,1.138355,disminuye_probabilidad
12,categorical__cod_trn_1388,1.033005,1.033005,aumenta_probabilidad
47,categorical__cod_trn_334,1.029400,1.029400,aumenta_probabilidad
55,categorical__cod_trn_574,-0.987026,0.987026,disminuye_probabilidad
34,categorical__cod_trn_2274,-0.926127,0.926127,disminuye_probabilidad


In [45]:
transaction_effects = coefficient_summary.loc[
    coefficient_summary["feature"].str.startswith(
        "categorical__cod_trn_"
    )
].copy()

transaction_effects["cod_trn"] = (
    transaction_effects["feature"]
    .str.replace(
        "categorical__cod_trn_",
        "",
        regex=False,
    )
)

train_transaction_data = pd.DataFrame(
    {
        "cod_trn": X_train["cod_trn"].astype(str),
        "target_paid_120d": y_train.to_numpy(),
    }
)

transaction_support = (
    train_transaction_data
    .groupby("cod_trn", as_index=False)
    .agg(
        records=("target_paid_120d", "size"),
        observed_payment_rate=(
            "target_paid_120d",
            "mean",
        ),
    )
)

transaction_effect_summary = (
    transaction_effects
    .merge(
        transaction_support,
        on="cod_trn",
        how="left",
    )
)

transaction_effect_summary["odds_ratio"] = np.exp(
    transaction_effect_summary["coefficient"]
)

transaction_effect_summary = (
    transaction_effect_summary
    .sort_values(
        "absolute_coefficient",
        ascending=False,
    )
)

transaction_effect_summary[
    [
        "cod_trn",
        "coefficient",
        "odds_ratio",
        "records",
        "observed_payment_rate",
        "effect",
    ]
].head(15)

,cod_trn,coefficient,odds_ratio,records,observed_payment_rate,effect
0,75,-2.313213,0.098943,25,0.000000,disminuye_probabilidad
1,416,-2.078583,0.125107,22,0.000000,disminuye_probabilidad
2,788,-1.249690,0.286594,78,0.230769,disminuye_probabilidad
3,301,-1.138355,0.320346,50,0.280000,disminuye_probabilidad
4,1388,1.033005,2.809497,84,0.750000,aumenta_probabilidad
5,334,1.029400,2.799384,29,0.758621,aumenta_probabilidad
6,574,-0.987026,0.372684,121,0.280992,disminuye_probabilidad
7,2274,-0.926127,0.396085,54,0.296296,disminuye_probabilidad
8,795,-0.860854,0.422801,55,0.327273,disminuye_probabilidad
9,321,0.858402,2.359387,25,0.720000,aumenta_probabilidad


## 5. Interpretación

Los códigos de transacción presentan diferencias relevantes, pero deben analizarse junto con su volumen y tasa observada. Los coeficientes indican asociaciones, no relaciones causales.

In [46]:
from sklearn.model_selection import GroupKFold, cross_val_predict


group_cross_validation = GroupKFold(n_splits=5)

oof_payment_probability = cross_val_predict(
    estimator=logistic_model,
    X=X,
    y=y,
    groups=groups,
    cv=group_cross_validation,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

model_data["payment_probability_120d_oof"] = (
    oof_payment_probability
)

model_data["expected_full_recovery_120d"] = (
    model_data["vlr_original"]
    * model_data["payment_probability_120d_oof"]
)

model_data["expected_non_full_recovery_exposure_120d"] = (
    model_data["vlr_original"]
    * (1 - model_data["payment_probability_120d_oof"])
)

In [47]:
model_data["risk_segment"] = pd.cut(
    model_data["payment_probability_120d_oof"],
    bins=[0.0, 0.40, 0.60, 0.80, 1.0],
    labels=[
        "alto_riesgo",
        "riesgo_medio",
        "probable_pago",
        "alta_probabilidad_pago",
    ],
    include_lowest=True,
)

risk_segment_validation = (
    model_data
    .groupby(
        "risk_segment",
        observed=False,
        as_index=False,
    )
    .agg(
        records=("cxc_id", "count"),
        original_amount=("vlr_original", "sum"),
        average_probability=(
            "payment_probability_120d_oof",
            "mean",
        ),
        observed_payment_rate=(
            "target_paid_120d",
            "mean",
        ),
        expected_full_recovery_120d=(
            "expected_full_recovery_120d",
            "sum",
        ),
        expected_non_full_recovery_exposure_120d=(
            "expected_non_full_recovery_exposure_120d",
            "sum",
        ),
    )
)

risk_segment_validation

,risk_segment,records,original_amount,average_probability,observed_payment_rate,expected_full_recovery_120d,expected_non_full_recovery_exposure_120d
0,alto_riesgo,9399,63877534.93,0.287641,0.329397,1.790554e+07,4.597200e+07
1,riesgo_medio,4952,32518531.84,0.500867,0.441438,1.646990e+07,1.604863e+07
2,probable_pago,5187,32384935.19,0.701764,0.664353,2.265612e+07,9.728819e+06
3,alta_probabilidad_pago,1738,8796496.56,0.837901,0.918872,7.331951e+06,1.464545e+06


## 6. Segmentación y uso de negocio

La tasa observada de pago aumenta entre los segmentos, lo que permite ordenar las obligaciones por riesgo. La exposición esperada ayuda a priorizar los casos que combinan menor probabilidad de pago y mayor valor económico.

Las probabilidades son estimaciones para apoyar la gestión, no decisiones automáticas.

## 7. Reproducibilidad

El notebook documenta la definición del objetivo, la comparación y la selección del modelo. La generación operativa del modelo, las métricas, las predicciones y el archivo para Power BI se realiza desde la raíz del proyecto con:

`python -m src.run_activity_2`

Esta separación evita duplicar la lógica de producción dentro del notebook.